# Danh gia cheo mien du lieu bo sung (Viec C, D, E)

Notebook nay CHI phuc vu muc dich danh gia (khong huan luyen lai bat ky model nao), nham hoan thien ma tran danh gia cheo 3x3 giua ba bo du lieu: **Vehicles-COCO**, **CanTho**, **AIC-HCMC-2020**.

Ba phep danh gia duoc thuc hien:
- **Viec C**: Model Vehicles-COCO (Baseline goc) danh gia tren tap test AIC-HCMC-2020
- **Viec D**: Model AIC-HCMC-2020 (cuoi chuoi, da qua 2 lan tinh chinh) danh gia tren tap test Vehicles-COCO
- **Viec E**: Model AIC-HCMC-2020 danh gia tren tap test CanTho

Ket qua cua notebook nay duoc dung lam phu luc bao cao, bo sung cho ma tran danh gia cheo da trinh bay tai muc 4.4.1.

## Buoc 1: Cai dat thu vien Ultralytics

In [2]:
!pip install ultralytics -q
print("Da cai dat xong ultralytics")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 6.8 MB/s eta 0:00:00
Da cai dat xong ultralytics


## Buoc 2: Ket noi Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## Buoc 3: Khai bao toan bo duong dan can dung

In [4]:
import os

# ---- Duong dan model (tren Google Drive, khong can copy ve local) ----
MODEL_COCO_PATH = "/content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline/YOLOv8n/baseline_100epochs_es/weights/best.pt"
MODEL_HCMC_PATH = "/content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline/finetune_hcmc_701515/weights/best.pt"

# ---- Duong dan dataset GOC tren Drive (se duoc copy ve local o Buoc 4) ----
DRIVE_DIR_COCO = "/content/drive/MyDrive/Do an AI giao thong/Dataset/Vehicles-COCO_701515"
DRIVE_DIR_HCMC = "/content/drive/MyDrive/Do an AI giao thong/Dataset/Vehicle Counting AIC HCMC 2020.v1i.yolov8_701515"

# ---- Duong dan dataset se copy ve, TREN O CUNG LOCAL cua Colab de danh gia nhanh va on dinh ----
LOCAL_DIR_COCO = "/content/coco_701515"
LOCAL_DIR_HCMC = "/content/hcmc_701515"

# ---- File yaml cua CanTho: dung THANG tren Drive (dataset nho, khong can copy) ----
DATA_YAML_CANTHO = "/content/drive/MyDrive/Do an AI giao thong/Dataset/CanTho_701515/data.yaml"

# ---- Thu muc luu ket qua xuat ra (project) ----
PROJECT_DIR = "/content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline"

print("Kiem tra su ton tai cua cac thanh phan chinh:")
print("  Model Vehicles-COCO :", os.path.exists(MODEL_COCO_PATH))
print("  Model AIC-HCMC-2020 :", os.path.exists(MODEL_HCMC_PATH))
print("  Dataset COCO (Drive):", os.path.exists(DRIVE_DIR_COCO))
print("  Dataset HCMC (Drive):", os.path.exists(DRIVE_DIR_HCMC))
print("  Yaml CanTho          :", os.path.exists(DATA_YAML_CANTHO))

Kiem tra su ton tai cua cac thanh phan chinh:
  Model Vehicles-COCO : True
  Model AIC-HCMC-2020 : True
  Dataset COCO (Drive): True
  Dataset HCMC (Drive): True
  Yaml CanTho          : True


## Buoc 4: Copy dataset Vehicles-COCO va AIC-HCMC-2020 ve o cung local

Viec copy ve local (thay vi doc truc tiep tu Google Drive) giup qua trinh danh gia
chay nhanh va on dinh hon, tranh hien tuong doc anh cham gay sai lech ket qua.
Neu gap loi "Transport endpoint is not connected" (loi ket noi Drive tam thoi),
chi can chay lai dung cell nay 1 lan nua.

In [5]:
import shutil, time

def copy_dataset_an_toan(drive_dir, local_dir, ten_hien_thi):
    if os.path.exists(local_dir):
        shutil.rmtree(local_dir)
    t0 = time.time()
    try:
        shutil.copytree(drive_dir, local_dir)
        print(f"[OK] Da copy dataset {ten_hien_thi} ve {local_dir} ({time.time()-t0:.1f}s)")
    except shutil.Error as e:
        print(f"[CANH BAO] Co {len(e.args[0])} file bi loi tam thoi khi copy {ten_hien_thi}.")
        print("           Kiem tra lai so luong anh o buoc sau, neu thieu nhieu thi chay lai cell nay.")

copy_dataset_an_toan(DRIVE_DIR_COCO, LOCAL_DIR_COCO, "Vehicles-COCO")
copy_dataset_an_toan(DRIVE_DIR_HCMC, LOCAL_DIR_HCMC, "AIC-HCMC-2020")

DATA_YAML_COCO_LOCAL = f"{LOCAL_DIR_COCO}/data.yaml"
DATA_YAML_HCMC_LOCAL = f"{LOCAL_DIR_HCMC}/data.yaml"

print()
print("File yaml COCO (local) ton tai:", os.path.exists(DATA_YAML_COCO_LOCAL))
print("File yaml HCMC (local) ton tai:", os.path.exists(DATA_YAML_HCMC_LOCAL))

[OK] Da copy dataset Vehicles-COCO ve /content/coco_701515 (774.0s)
[OK] Da copy dataset AIC-HCMC-2020 ve /content/hcmc_701515 (719.9s)

File yaml COCO (local) ton tai: True
File yaml HCMC (local) ton tai: True


## Buoc 5: Kiem tra lai so luong anh da copy (doi chieu voi so lieu goc)

In [6]:
def dem_anh(local_dir, ten_hien_thi, ky_vong=None):
    print(f"--- {ten_hien_thi} ---")
    for split in ["train", "valid", "test"]:
        img_dir = f"{local_dir}/{split}/images"
        if os.path.exists(img_dir):
            so_luong = len(os.listdir(img_dir))
            print(f"  {split}: {so_luong} anh")
        else:
            print(f"  {split}: KHONG TON TAI")
    print()

# So lieu goc da biet: Vehicles-COCO (Train 13.298 / Valid 2.849 / Test 2.851)
dem_anh(LOCAL_DIR_COCO, "Vehicles-COCO")
# So lieu goc da biet: AIC-HCMC-2020, tap test co 2.267 anh
dem_anh(LOCAL_DIR_HCMC, "AIC-HCMC-2020")

--- Vehicles-COCO ---
  train: 13298 anh
  valid: 2849 anh
  test: 2851 anh

--- AIC-HCMC-2020 ---
  train: 10572 anh
  valid: KHONG TON TAI
  test: 2267 anh



## Viec C: Model Vehicles-COCO (Baseline goc) danh gia tren tap test AIC-HCMC-2020

Muc dich: kiem tra model goc, chua tung tiep xuc du lieu Viet Nam, khi gap du lieu
giao thong TP.HCM thi nhan dien duoc o muc do nao. Day la diem doi chung nen truoc
khi tinh chinh.

In [7]:
from ultralytics import YOLO

model_coco = YOLO(MODEL_COCO_PATH)

metrics_c = model_coco.val(
    data=DATA_YAML_HCMC_LOCAL,
    split="test",
    imgsz=640,
    batch=16,
    project=PROJECT_DIR,
    name="eval_coco_on_hcmc_test",
    exist_ok=True,
)

print("===== VIEC C: MODEL VEHICLES-COCO - TREN TAP TEST AIC-HCMC-2020 =====")
print(f"mAP@0.5      : {metrics_c.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {metrics_c.box.map:.4f}")
print(f"Precision    : {metrics_c.box.mp:.4f}")
print(f"Recall       : {metrics_c.box.mr:.4f}")

print("\nmAP@0.5 theo tung lop:")
names_c = model_coco.names
for i in range(len(metrics_c.box.ap_class_index)):
    c = metrics_c.box.ap_class_index[i]
    print(f"  {names_c[c]:12s} -> {metrics_c.box.ap50[i]:.4f}")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 14.7±29.2 ms, read: 39.2±33.4 MB/s, size: 57.4 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/MyDrive/Do an AI giao thong/Dataset/Vehicle Counting AIC HCMC 2020.v1i.yolov8_701515/test/labels.cache... 2267 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2267/2267 365.7Mit/s 0.0s
val: /content/drive/MyDrive/Do an AI giao thong/Dataset/Vehicle Counti

## Viec D: Model AIC-HCMC-2020 danh gia tren tap test Vehicles-COCO

Muc dich: model da qua 2 lan tinh chinh lien tiep (COCO -> CanTho -> HCMC), kiem tra
xem sau khi hoc them 2 buoc, model co con nho duoc bai hoc DAU TIEN (Vehicles-COCO)
hay khong, hay da quen sach. Day la phep kiem tra "tri nho dai han" cua chuoi tinh
chinh tuan tu.

In [8]:
model_hcmc = YOLO(MODEL_HCMC_PATH)

metrics_d = model_hcmc.val(
    data=DATA_YAML_COCO_LOCAL,
    split="test",
    imgsz=640,
    batch=16,
    project=PROJECT_DIR,
    name="eval_hcmc_on_coco_test",
    exist_ok=True,
)

print("===== VIEC D: MODEL AIC-HCMC-2020 - TREN TAP TEST VEHICLES-COCO =====")
print(f"mAP@0.5      : {metrics_d.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {metrics_d.box.map:.4f}")
print(f"Precision    : {metrics_d.box.mp:.4f}")
print(f"Recall       : {metrics_d.box.mr:.4f}")

print("\nmAP@0.5 theo tung lop:")
names_d = model_hcmc.names
for i in range(len(metrics_d.box.ap_class_index)):
    c = metrics_d.box.ap_class_index[i]
    print(f"  {names_d[c]:12s} -> {metrics_d.box.ap50[i]:.4f}")

Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.5±0.1 ms, read: 50.5±25.7 MB/s, size: 58.3 KB)
val: Scanning /content/drive/MyDrive/Do an AI giao thong/Dataset/Vehicles-COCO_701515/test/labels.cache... 2851 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2851/2851 797.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 179/179 4.3it/s 41.6s
                   all       2851      10683      0.205      0.161     0.0843      0.042
                   bus        656        973      0.215      0.279      0.141       0.08
                   car       1937       6901       0.41      0.122      0.115     0.0595
              motobyte        531       1245      0.109       0.12     0.0466     0.0137
                 truck        940       1564     0.0859      0.124     0.0346  

## Viec E: Model AIC-HCMC-2020 danh gia tren tap test CanTho

Muc dich: kiem tra xem sau khi tinh chinh them buoc thu 3 (HCMC), model co con nho
duoc bai hoc LIEN TRUOC DO (CanTho) hay khong. Ket hop voi Viec D, day la 2 phep do
quan trong nhat de xac dinh hien tuong quen kien thuc co "tich luy nang them" qua
nhieu buoc tinh chinh lien tiep hay khong.

In [9]:
# model_hcmc da duoc nap o cell Viec D, dung lai khong can nap lai
metrics_e = model_hcmc.val(
    data=DATA_YAML_CANTHO,
    split="test",
    imgsz=640,
    batch=16,
    project=PROJECT_DIR,
    name="eval_hcmc_on_cantho_test",
    exist_ok=True,
)

print("===== VIEC E: MODEL AIC-HCMC-2020 - TREN TAP TEST CANTHO =====")
print(f"mAP@0.5      : {metrics_e.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {metrics_e.box.map:.4f}")
print(f"Precision    : {metrics_e.box.mp:.4f}")
print(f"Recall       : {metrics_e.box.mr:.4f}")

print("\nmAP@0.5 theo tung lop:")
names_e = model_hcmc.names
for i in range(len(metrics_e.box.ap_class_index)):
    c = metrics_e.box.ap_class_index[i]
    print(f"  {names_e[c]:12s} -> {metrics_e.box.ap50[i]:.4f}")

Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
WARNING ⚠️ val: Slow image access detected (ping: 1.1±0.9 ms, read: 4.6±9.9 MB/s, size: 42.5 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/MyDrive/Do an AI giao thong/Dataset/CanTho_701515/test/labels.cache... 167 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 167/167 28.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 1.8it/s 6.1s
                   all        167        980      0.669      0.694       0.75      0.529
                   bus          6          7      0.393      0.714      0.578      0.493
                   car        149        494      0.848      0.669      0.866      0.627
              motobyte        155        441      0.686      0.765      0.825      0.454
                 tru

## Bang tong hop ket qua Viec C, D, E

In [10]:
print("="*70)
print("BANG TONG HOP - HOAN THIEN MA TRAN DANH GIA CHEO 3x3")
print("="*70)
print(f"{'Phep danh gia':45s} {'mAP@0.5':>10s} {'mAP@0.5:0.95':>14s}")
print("-"*70)
print(f"{'Viec C: COCO model -> test HCMC':45s} {metrics_c.box.map50:>10.4f} {metrics_c.box.map:>14.4f}")
print(f"{'Viec D: HCMC model -> test COCO':45s} {metrics_d.box.map50:>10.4f} {metrics_d.box.map:>14.4f}")
print(f"{'Viec E: HCMC model -> test CanTho':45s} {metrics_e.box.map50:>10.4f} {metrics_e.box.map:>14.4f}")
print("="*70)
print()
print("Cac ket qua nay, ket hop voi cac gia tri da co truoc do, hoan thien")
print("day du ma tran danh gia cheo 3x3 giua Vehicles-COCO, CanTho va AIC-HCMC-2020.")

BANG TONG HOP - HOAN THIEN MA TRAN DANH GIA CHEO 3x3
Phep danh gia                                    mAP@0.5   mAP@0.5:0.95
----------------------------------------------------------------------
Viec C: COCO model -> test HCMC                   0.3731         0.2051
Viec D: HCMC model -> test COCO                   0.0843         0.0420
Viec E: HCMC model -> test CanTho                 0.7504         0.5286

Cac ket qua nay, ket hop voi cac gia tri da co truoc do, hoan thien
day du ma tran danh gia cheo 3x3 giua Vehicles-COCO, CanTho va AIC-HCMC-2020.
